In [1]:
doTraining = False  # <-- metti True quando vuoi ri-allenare
doStageC = True  # <-- opzionale: mix gold+silver dopo stage B


## Setup and Imports

In [2]:
import json
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

from seqeval.metrics import precision_score, recall_score, f1_score

torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


###  Define label space (entity types + BIO tagging)
 definition of the 13 GutBrainIE entity categories and expansions into BIO tags:
 - "O" for tokens outside any entity
 - "B-<label>" for the first token of an entity mention
 - "I-<label>" for continuation tokens
 Then we build `label2id` / `id2label` mappings so the model can train and decode labels.

 Finally we set:
 - the pretrained backbone (BioBERT)
 - the output directory where the fine-tuned model will be saved.

In [3]:
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]

label_list = ["O"]
for lab in ENTITY_LABELS:
    label_list.append(f"B-{lab}")
    label_list.append(f"I-{lab}")

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}

print("Total labels:", len(label_list))
print("First 10 labels:", label_list[:10])

model_name = "dmis-lab/biobert-v1.1"
output_model_dir = "models/bert_biomedbert_ner_twopass_2025to2026"  # nuovo nome

print("Model:", model_name)
print("Output directory:", output_model_dir)


Total labels: 27
First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']
Model: dmis-lab/biobert-v1.1
Output directory: models/bert_biomedbert_ner_twopass_2025to2026


### Init tokenizer + collato

In [4]:
print("Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

print("✓ Tokenizer + collator ready")


Initializing tokenizer...
✓ Tokenizer + collator ready


## Data loading + split (2025/2026)

In [5]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_2025 = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2025" / "Annotations"
DATA_2026 = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2026" / "Annotations"

print("DATA_2025:", DATA_2025)
print("DATA_2026:", DATA_2026)


def load_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)


def safe_load_ner(paths):
    data = {}
    for p in paths:
        p = Path(p)
        if not p.exists():
            print(f"[SKIP] Missing: {p}")
            continue
        tmp = load_json(p)
        data.update(tmp)
        print(f"[OK] Loaded {len(tmp)} docs from {p.name}")
    return data


train_2025_files = [
    DATA_2025 / "Train" / "platinum_quality" / "json_format" / "train_platinum.json",
    DATA_2025 / "Train" / "gold_quality" / "json_format" / "train_gold.json",
]

train_2026_gold_files = [
    DATA_2026 / "Train" / "gold_quality" / "json_format" / "train_gold.json",
]
train_2026_silver_files = [
    DATA_2026 / "Train" / "silver_quality" / "json_format" / "train_silver.json",
]

dev_2026_path = DATA_2026 / "Dev" / "json_format" / "dev.json"
dev_2026_data = load_json(dev_2026_path)
dev_data = dev_2026_data
print("Dev 2026 docs:", len(dev_2026_data))


DATA_2025: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2025\Annotations
DATA_2026: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations
Dev 2026 docs: 80


### Prepare docs (title/abstract)

In [6]:
def prepare_documents_for_ner(data):
    documents = []
    for pmid, article in data.items():
        title_text = article["metadata"]["title"]
        title_entities = [e for e in article["entities"] if e["location"] == "title"]
        documents.append({"pmid": pmid, "location": "title", "text": title_text, "entities": title_entities})

        abstract_text = article["metadata"]["abstract"]
        abstract_entities = [e for e in article["entities"] if e["location"] == "abstract"]
        documents.append({"pmid": pmid, "location": "abstract", "text": abstract_text, "entities": abstract_entities})
    return documents


dev_documents = prepare_documents_for_ner(dev_2026_data)
print("Dev 2026 segments:", len(dev_documents))


Dev 2026 segments: 160


## BIO alignment

In [7]:
def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    offsets = enc["offset_mapping"]

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    labels = ["O"] * len(input_ids)

    # sort: start asc, length desc
    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1  # inclusive -> exclusive
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, (ts, te) in enumerate(offsets):
            ts = int(ts);
            te = int(te)
            if ts == 0 and te == 0:
                continue
            if ts < ent_end_excl and te > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        if ent_token_start is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue
                tag = f"B-{ent_label}" if i == ent_token_start else f"I-{ent_label}"
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(t, label2id["O"]) for t in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
    }


### Dataset class + processor

In [8]:
class NERDataset(Dataset):
    def __init__(self, processed):
        self.data = processed

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        return {
            "input_ids": torch.tensor(x["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(x["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(x["labels"], dtype=torch.long),
        }


def build_processed_segments(raw_data_dict):
    docs = prepare_documents_for_ner(raw_data_dict)
    processed = []
    for doc in tqdm(docs, desc="Align BIO"):
        ex = align_labels_with_tokens(doc["text"], doc["entities"], tokenizer, label2id, max_length=512)
        ex["pmid"] = doc["pmid"]
        ex["location"] = doc["location"]
        ex["text"] = doc["text"]
        ex["entities"] = doc["entities"]
        processed.append(ex)
    return processed



### Build dev data

In [9]:
processed_dev = []
for doc in tqdm(dev_documents, desc="Processing dev 2026"):
    ex = align_labels_with_tokens(doc["text"], doc["entities"], tokenizer, label2id, max_length=512)
    ex["pmid"] = doc["pmid"]
    ex["location"] = doc["location"]
    ex["text"] = doc["text"]
    ex["entities"] = doc["entities"]
    processed_dev.append(ex)

dev_dataset = NERDataset(processed_dev)
print("Processed dev segments:", len(processed_dev))


Processing dev 2026: 100%|██████████| 160/160 [00:00<00:00, 473.77it/s]

Processed dev segments: 160


### Metrics + weighted trainer

In [10]:
def compute_metrics_seqeval(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(preds, labels):
        seq_true = []
        seq_pred = []
        for p_id, l_id in zip(pred_seq, label_seq):
            if int(l_id) == -100:
                continue
            seq_true.append(id2label[int(l_id)])
            seq_pred.append(id2label[int(p_id)])
        true_labels.append(seq_true)
        true_preds.append(seq_pred)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


def compute_class_weights(processed_train, num_labels, ignore_index=-100, power=0.5):
    counts = Counter()
    for ex in processed_train:
        for y in ex["labels"]:
            if y == ignore_index:
                continue
            counts[int(y)] += 1

    freqs = np.zeros(num_labels, dtype=np.float64)
    for c in range(num_labels):
        freqs[c] = counts.get(c, 0)
    freqs[freqs == 0] = 1.0

    weights = (1.0 / freqs) ** power
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float)


class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100,
        )
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


print("✓ Metrics + WeightedLossTrainer ready")


✓ Metrics + WeightedLossTrainer ready


### TRAINING multi-stage
only if doTraining =True

In [11]:
from collections import Counter
ckpt_stageA_dir = output_model_dir + "_stageA_2025hq"
ckpt_stageB_dir = output_model_dir + "_stageB_2026gold"
ckpt_stageC_dir = output_model_dir + "_stageC_mix"

if doTraining:
    # ---------------------------
    # STAGE A: 2025 platinum+gold
    # ---------------------------
    train_2025_data = safe_load_ner(train_2025_files)
    processed_train_2025 = build_processed_segments(train_2025_data)
    train_dataset_2025 = NERDataset(processed_train_2025)

    class_weights_2025 = compute_class_weights(processed_train_2025, num_labels=len(label_list), power=0.5)
    class_weights_2025 = torch.clamp(class_weights_2025, min=0.5, max=5.0)

    args_stageA = TrainingArguments(
        output_dir=ckpt_stageA_dir,
        learning_rate=3e-5,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=100,
        save_total_limit=2,
        seed=42,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    model_stageA = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )

    trainer_stageA = WeightedLossTrainer(
        model=model_stageA,
        args=args_stageA,
        train_dataset=train_dataset_2025,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_seqeval,
        class_weights=class_weights_2025,
    )

    print("=== STAGE A: Pretrain 2025 HQ ===")
    trainer_stageA.train()
    trainer_stageA.save_model(ckpt_stageA_dir)
    tokenizer.save_pretrained(ckpt_stageA_dir)

    # ---------------------------
    # STAGE B: 2026 gold
    # ---------------------------
    train_2026_gold_data = safe_load_ner(train_2026_gold_files)
    processed_train_2026_gold = build_processed_segments(train_2026_gold_data)
    train_dataset_2026_gold = NERDataset(processed_train_2026_gold)

    class_weights_2026_gold = compute_class_weights(processed_train_2026_gold, num_labels=len(label_list), power=0.5)
    class_weights_2026_gold = torch.clamp(class_weights_2026_gold, min=0.5, max=5.0)

    args_stageB = TrainingArguments(
        output_dir=ckpt_stageB_dir,
        learning_rate=2e-5,
        lr_scheduler_type="linear",
        warmup_ratio=0.05,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=2,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=100,
        save_total_limit=2,
        seed=42,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    model_stageB = AutoModelForTokenClassification.from_pretrained(ckpt_stageA_dir)

    trainer_stageB = WeightedLossTrainer(
        model=model_stageB,
        args=args_stageB,
        train_dataset=train_dataset_2026_gold,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_seqeval,
        class_weights=class_weights_2026_gold,
    )

    print("=== STAGE B: Finetune 2026 GOLD ===")
    trainer_stageB.train()
    trainer_stageB.save_model(ckpt_stageB_dir)
    tokenizer.save_pretrained(ckpt_stageB_dir)

    # ---------------------------
    # STAGE C (optional): mix gold+silver
    # ---------------------------
    if doStageC:
        train_2026_silver_data = safe_load_ner(train_2026_silver_files)
        processed_train_2026_silver = build_processed_segments(train_2026_silver_data)

        processed_mix = (processed_train_2026_gold * 3) + processed_train_2026_silver
        np.random.shuffle(processed_mix)

        train_dataset_mix = NERDataset(processed_mix)

        class_weights_mix = compute_class_weights(processed_mix, num_labels=len(label_list), power=0.5)
        class_weights_mix = torch.clamp(class_weights_mix, min=0.5, max=5.0)

        args_stageC = TrainingArguments(
            output_dir=ckpt_stageC_dir,
            learning_rate=2e-5,
            lr_scheduler_type="linear",
            warmup_ratio=0.03,
            weight_decay=0.01,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            gradient_accumulation_steps=2,
            num_train_epochs=1,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            logging_steps=100,
            save_total_limit=2,
            seed=42,
            fp16=torch.cuda.is_available(),
            report_to="none",
        )

        model_stageC = AutoModelForTokenClassification.from_pretrained(ckpt_stageB_dir)

        trainer_stageC = WeightedLossTrainer(
            model=model_stageC,
            args=args_stageC,
            train_dataset=train_dataset_mix,
            eval_dataset=dev_dataset,
            data_collator=data_collator,
            compute_metrics=compute_metrics_seqeval,
            class_weights=class_weights_mix,
        )

        print("=== STAGE C: Mix 2026 GOLD + SILVER ===")
        trainer_stageC.train()
        trainer_stageC.save_model(ckpt_stageC_dir)
        tokenizer.save_pretrained(ckpt_stageC_dir)

# ---------------------------
# Decide final checkpoint
# ---------------------------
final_model_dir = ckpt_stageC_dir if (doStageC and Path(ckpt_stageC_dir).exists()) else ckpt_stageB_dir
if not Path(final_model_dir).exists():
    # fallback: se non hai mai allenato, usa base model (ma ti conviene allenare almeno stage B)
    final_model_dir = model_name

print("FINAL MODEL DIR:", final_model_dir)

inference_model = AutoModelForTokenClassification.from_pretrained(final_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(final_model_dir)
inference_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)

print("✓ Inference model loaded")
print("✓ Device:", device)


FINAL MODEL DIR: models/bert_biomedbert_ner_twopass_2025to2026_stageC_mix


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 447.02it/s, Materializing param=classifier.weight]                                      


✓ Inference model loaded
✓ Device: cuda


# INFERENCE

### Inference setup (load again + select device)

In [12]:
import json
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoModelForTokenClassification, AutoTokenizer

print("Loading model for inference...")
inference_model = AutoModelForTokenClassification.from_pretrained(final_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(final_model_dir)
inference_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)

print("✓ Model loaded from:", final_model_dir)
print("✓ Device:", device)


Loading model for inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 488.70it/s, Materializing param=classifier.weight]                                      


✓ Model loaded from: models/bert_biomedbert_ner_twopass_2025to2026_stageC_mix
✓ Device: cuda



##  Two-pass threshold strategy (precision first, recall second)
 Instead of taking all decoded entities, we filter them by label-specific confidence thresholds.
 The idea:
 - Pass 1 uses strict thresholds to **keep only high-precision mentions**.
 - Pass 2 relaxes thresholds only for selected labels where recall is typically low.
#- Later **we merge the two passe**s while preventing overlaps with pass-1 outputs.

 This is a post-processing policy that can improve macro-F1 by balancing precision/recall per label.

In [13]:
# Pass 1: your best (high precision)
LABEL_THRESH_HIGH = {
    "DDF": 0.88,
    "bacteria": 0.88,
    "statistical technique": 0.92,
    "biomedical technique": 0.82,
    "gene": 0.75,
    "food": 0.70,
    "chemical": 0.80,
    "dietary supplement": 0.85,
    "drug": 0.80,
    "microbiome": 0.78,
    "anatomical location": 0.78,
    "human": 0.70,
    "animal": 0.70,
}

LABEL_THRESH_HIGH.update({
    "chemical": 0.72,
    "food": 0.60,
    "dietary supplement": 0.75,
    "gene": 0.68,
    "bacteria": 0.84,
    "biomedical technique": 0.78,
    "statistical technique": 0.88,
})
#
# LABEL_THRESH_RECALL.update({
#     "chemical": 0.62,
#     "food": 0.45,
#     "dietary supplement": 0.60,
#     "gene": 0.58,
#     "bacteria": 0.78,
#     "biomedical technique": 0.72,
#     "statistical technique": 0.84,
# })
LABEL_THRESH_RECALL={
    "chemical": 0.65,
    "bacteria": 0.78,
    "biomedical technique": 0.72,
    "food": 0.50,
    "gene": 0.65,
    "dietary supplement": 0.72,
}

RECALL_LABELS = {"chemical","food","dietary supplement","gene","bacteria","biomedical technique","statistical technique"}


DEFAULT_THRESH = 0.80


### Simple false-positive filters + label-specific postprocessing
This cell defines lightweight heuristics to remove obvious junk predictions:
- generic terms that are too unspecific (e.g., "microbes")
- markup fragments ("<...>")
- too short spans

It also adds a targeted postprocessing rule:
 - If something looks gene-like (e.g., IL-6, TNF-α, α-synuclein) but was predicted as "chemical",  remap it to "gene". This fixes a common confusion pattern that we observed.

In [14]:
import re

BAD_BACTERIA = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}
BAD_CHEMICAL = {"metabolites", "neurotransmitters"}
BAD_DIETSUPP = {"nnss", "sp", "fep", "ns9", "pro"}

BAD_MICROBIOME = {"micro", "microbiota", "gut"}

DIET_CONCEPT = {
    "diet", "ketogenic diet", "high-fat diet", "high fat diet",
    "high glycemic diet", "vegetarian diet", "balanced diet",
    "western diet", "mediterranean diet"
}
BAD_FOOD_EXACT = {
    "control", "ketogenic", "high-fat", "high fat", "high",
    "glycemic index", "lycemic index",
    "food", "ingested food"
}


def normalize_span(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def apply_simple_filters(entities):
    out = []
    for e in entities:
        s = normalize_span(e.get("text_span", ""))

        if not s:
            continue
        if "<" in s or ">" in s:
            continue
        if len(s) <= 1:
            continue

        lab = e["label"]
        if lab == "bacteria" and s in BAD_BACTERIA:
            continue
        if lab == "dietary supplement" and s in BAD_DIETSUPP:
            continue
        if lab == "chemical" and s in BAD_CHEMICAL:
            continue
        if lab == "microbiome" and s in BAD_MICROBIOME:
            continue
        if lab == "food":
            if s in DIET_CONCEPT or s.endswith(" diet"):
                continue
            if s in BAD_FOOD_EXACT:
                continue

        out.append(e)
    return out


GENE_LIKE = re.compile(
    r"^(il-\d+|tnf(-?α)?|ifn(-?γ)?|tgf(-?β\d*)?|snca|park7|dj-1|mapt|apoe\d*|hla-[a-z0-9\*\:]+)$",
    re.IGNORECASE,
)
CHEM_LIKE = re.compile(
    r"(aβ|amyloid|scfa|gaba|succinate|butyrate|propionate|acetate|\b[a-z]+ate\b|\b[a-z]+acid\b|\(\d+\-\d+\))",
    re.IGNORECASE,
)


def postprocess_gene_vs_chemical(entities):
    for e in entities:
        s = normalize_span(e.get("text_span", ""))
        if e["label"] == "chemical" and GENE_LIKE.match(s):
            e["label"] = "gene"
        elif e["label"] == "gene" and CHEM_LIKE.search(s):
            e["label"] = "chemical"
    return entities


FOOD_ANCHORS = {"kefir", "yogurt", "milk", "cheese", "cookie", "lentil", "lentils", "buckwheat", "wheat", "rice", "tea",
                "coffee"}
SUPP_HARD = {"capsule", "tablet", "extract", "powder"}
SUPP_SOFT = {"probiotic", "probiotics", "prebiotic", "prebiotics", "synbiotic", "synbiotics", "supplement"}


def postprocess_food_vs_supp(entities):
    for e in entities:
        s = normalize_span(e.get("text_span", ""))

        if any(w in s for w in FOOD_ANCHORS):
            if e["label"] in {"dietary supplement", "food"}:
                e["label"] = "food"
            continue

        if any(w in s for w in SUPP_HARD):
            if e["label"] in {"dietary supplement", "food"}:
                e["label"] = "dietary supplement"
            continue

        if e["label"] == "food" and any(w in s for w in SUPP_SOFT):
            e["label"] = "dietary supplement"
    return entities


### Core predictor: decode BIO + compute entity confidence score
This is the main inference function that:
1) tokenizes text with offsets
2) runs the model to get logits -> softmax probabilities
3) converts per-token predictions to BIO labels
4) rebuilds entity spans by scanning tokens left-to-right

**Scoring:**
 - For each entity we compute a confidence score as the mean token probability
 over the entity span (B-tag prob for first token + I-tag prob for continuation tokens).

**BIO repair:**
- If we see I-X without an active entity, we start a new entity (treat as B-X).
- If we see I-X but we are currently inside Y, we close Y and start X.

 These rules make decoding more robust to occasional BIO inconsistencies

In [15]:
def predict_entities_with_scores(
        model,
        tokenizer,
        text: str,
        id2label: dict,
        label2id: dict,
        device,
        max_length: int = 512,
):
    if not text:
        return []

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=max_length,
    )

    offsets = enc.pop("offset_mapping")[0].cpu().numpy()  # (T,2)
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        logits = out.logits[0]  # [T, C]
        probs = torch.softmax(logits, dim=-1)  # [T, C]
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
        probs_cpu = probs.cpu().numpy()

    tags = [id2label[int(i)] for i in pred_ids]

    entities = []
    current = None

    def _start(ent_label: str, s: int, e_excl: int, t_idx: int):
        b_idx = label2id.get(f"B-{ent_label}")
        tok_prob = float(probs_cpu[t_idx, b_idx]) if b_idx is not None else float(probs_cpu[t_idx].max())
        return {
            "start_idx": int(s),
            "end_idx": int(e_excl - 1),  # store inclusive
            "label": ent_label,
            "text_span": text[s:e_excl],
            "_token_probs": [tok_prob],
        }

    def _extend(ent: dict, e_excl: int, t_idx: int):
        lab = ent["label"]
        i_idx = label2id.get(f"I-{lab}")
        tok_prob = float(probs_cpu[t_idx, i_idx]) if i_idx is not None else float(probs_cpu[t_idx].max())
        ent["end_idx"] = int(e_excl - 1)
        ent["text_span"] = text[ent["start_idx"]:e_excl]
        ent["_token_probs"].append(tok_prob)

    for t_idx, (tag, (s, e)) in enumerate(zip(tags, offsets)):
        s = int(s);
        e = int(e)

        # special tokens
        if s == 0 and e == 0:
            continue
        if e <= s:
            continue

        if tag.startswith("B-"):
            if current is not None:
                entities.append(current)
            current = _start(tag[2:], s, e, t_idx)

        elif tag.startswith("I-"):
            lab = tag[2:]
            if current is None:
                current = _start(lab, s, e, t_idx)  # BIO repair
            elif lab != current["label"]:
                entities.append(current)
                current = _start(lab, s, e, t_idx)  # BIO repair
            else:
                _extend(current, e, t_idx)

        else:  # O
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    for ent in entities:
        probs_list = ent.pop("_token_probs", [])

        if not probs_list:
            ent["score"] = 0.0
            continue

        # ✅ più recall-friendly del mean
        # opzione A: median
        ent["score"] = float(np.median(probs_list))

        # opzione B (alternativa): mean dei top-2 token (ancora più permissiva)
        # ent["score"] = float(np.mean(sorted(probs_list, reverse=True)[:2]))

    return entities


###  Threshold filtering (label -> threshold map)
 This helper keeps an entity only if:
- its computed score >= the threshold configured for its label
-  Labels not explicitly listed use `DEFAULT_THRESH`

In [16]:
def passes_threshold(e, label_thresh):
    thr = label_thresh.get(e["label"], DEFAULT_THRESH)
    s = normalize_span(e["text_span"])

    # food: se c'è "diet" alza la soglia (evita FP tipo "ketogenic diet")
    if e["label"] == "food" and ("diet" in s):
        thr = max(thr, 0.70)

    return e.get("score", 0.0) >= thr


def filter_by_threshold_with_map(entities, label_thresh):
    return [e for e in entities if passes_threshold(e, label_thresh)]


### Merge policy (no overlap with pass-1)
We merge two entity lists (pass-1 and pass-2) with a conservative rule:
- Keep all pass-1 (high precision)
- Add pass-2 entities only for selected labels (`RECALL_LABELS`)
- Add only if their character span does not overlap any already kept entity in the same segment
This avoids creating duplicated/competing spans that often hurt precision.

In [17]:
def span_iou(a, b):
    inter = max(0, min(a["end_idx"], b["end_idx"]) - max(a["start_idx"], b["start_idx"]) + 1)
    if inter == 0:
        return 0.0
    la = a["end_idx"] - a["start_idx"] + 1
    lb = b["end_idx"] - b["start_idx"] + 1
    return inter / (la + lb - inter)


def any_overlap(ent, kept, iou_thr=0.5):
    for k in kept:
        if k["location"] != ent["location"]:
            continue
        if ent["label"] == "food":
            # blocca solo se è quasi identico (doppione vero)
            if span_iou(ent, k) >= 0.85 and ent["label"] == k["label"]:
                return True
            continue

        if span_iou(ent, k) >= iou_thr:
            return True
    return False

def merge_two_pass(ents_high, ents_rec, recall_labels):
    kept = list(ents_high)
    for e in ents_rec:
        if e["label"] not in recall_labels:
            continue
        if not any_overlap(e, kept):
            kept.append(e)
    return kept


In [18]:
def overlaps(a, b):
    return not (a["end_idx"] < b["start_idx"] or a["start_idx"] > b["end_idx"])

def overlap_ratio(a, b):
    inter = max(0, min(a["end_idx"], b["end_idx"]) - max(a["start_idx"], b["start_idx"]) + 1)
    if inter == 0:
        return 0.0
    la = a["end_idx"] - a["start_idx"] + 1
    lb = b["end_idx"] - b["start_idx"] + 1
    return inter / min(la, lb)  # overlap rispetto al più corto

def dedup_segment(entities):
    seen = set()
    out = []
    for e in entities:
        key = (e["start_idx"], e["end_idx"], e["location"], e["label"])
        if key in seen:
            continue
        seen.add(key)
        out.append(e)
    return out

def soft_overlap_prune(entities, same_label_only=True, ratio_thr=0.85):
    """
    Rimuove solo duplicati quasi identici.
    - Se same_label_only=True: pruna overlap solo se stessa label.
    - ratio_thr=0.85: overlap quasi totale del più corto.
    Tiene quello con score più alto se presente, altrimenti il più lungo.
    """
    ents = sorted(entities, key=lambda x: (x["location"], x["start_idx"], -(x["end_idx"]-x["start_idx"])))
    kept = []

    for e in ents:
        drop = False
        for i, k in enumerate(kept):
            if k["location"] != e["location"]:
                continue
            if not overlaps(k, e):
                continue

            if same_label_only and k["label"] != e["label"]:
                continue

            if overlap_ratio(k, e) >= ratio_thr:
                # scegli il migliore
                ks = k.get("score", None)
                es = e.get("score", None)

                if (ks is not None) and (es is not None):
                    better = e if es > ks else k
                else:
                    len_k = k["end_idx"] - k["start_idx"]
                    len_e = e["end_idx"] - e["start_idx"]
                    better = e if len_e > len_k else k

                kept[i] = better
                drop = True
                break

        if not drop:
            kept.append(e)

    return kept


### Two-pass segment predictor + dev inference loop
 This function wraps the full inference pipeline for one text segment:
 - decode entities with scores
 - pass 1: strict thresholds + filters + postprocessing
 - pass 2: relaxed thresholds (selected labels) + filters + postprocessing
 - merge with "no overlap with pass-1" policy
 - remove the score field so the output matches submission schema

 Then we run it across all dev segments and aggregate entities back by PMID.

In [19]:
TRIM_CHARS = " \t\n\r.,;:()[]{}<>\"'"

def trim_entity_span(e, text):
    s = int(e["start_idx"])
    end = int(e["end_idx"])

    # safe bounds
    s = max(0, min(s, len(text)))
    end = max(0, min(end, len(text)-1))

    # trim left
    while s <= end and text[s] in TRIM_CHARS:
        s += 1
    # trim right
    while end >= s and text[end] in TRIM_CHARS:
        end -= 1

    if s <= end:
        e["start_idx"] = s
        e["end_idx"] = end
        e["text_span"] = text[s:end+1]
    return e


Segment predictor

In [20]:
def predict_segment_entities_two_pass(text, location):
    ents_raw = predict_entities_with_scores(
        model=inference_model,
        tokenizer=inference_tokenizer,
        text=text,
        id2label=id2label,
        label2id=label2id,
        device=device,
        max_length=512,
    )

    # 1) trim + add location early
    trimmed = []
    for e in ents_raw:
        e2 = trim_entity_span(e, text)
        if e2 is None:
            continue
        if not e2.get("text_span"):
            continue
        e2["location"] = location          # ✅ IMPORTANT
        trimmed.append(e2)

    # 2) postprocess BEFORE threshold
    trimmed_pp = postprocess_gene_vs_chemical(trimmed)
    trimmed_pp = postprocess_food_vs_supp(trimmed_pp)

    # 3) pass 1
    ents_high = filter_by_threshold_with_map(trimmed_pp, LABEL_THRESH_HIGH)
    ents_high = apply_simple_filters(ents_high)

    # 4) pass 2
    ents_rec = filter_by_threshold_with_map(trimmed_pp, LABEL_THRESH_RECALL)
    ents_rec = apply_simple_filters(ents_rec)

    # 5) merge
    merged = merge_two_pass(ents_high, ents_rec, recall_labels=RECALL_LABELS)

    # 6) dedup + soft overlap prune (expects location present ✅)
    merged = dedup_segment(merged)
    merged = soft_overlap_prune(merged, same_label_only=True, ratio_thr=0.85)

    # 7) remove score for submission
    for e in merged:
        e.pop("score", None)

    return merged


## Run Inference + Save Predictions

In [21]:
print("Running inference on dev (two-pass)...")

predictions_two_pass = {}

for doc in tqdm(dev_documents, desc="Predicting"):
    pmid = doc["pmid"]
    location = doc["location"]
    text = doc["text"]

    ents = predict_segment_entities_two_pass(text, location)

    predictions_two_pass.setdefault(pmid, {"entities": []})
    predictions_two_pass[pmid]["entities"].extend(ents)

print("✓ Inference completed docs:", len(predictions_two_pass))
print("Total predicted entities:", sum(len(v["entities"]) for v in predictions_two_pass.values()))

# Save
output_path = r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_two_pass_2025to2026.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(predictions_two_pass, f, ensure_ascii=False, indent=2)

print("Saved predictions to:", output_path)


Running inference on dev (two-pass)...


Predicting: 100%|██████████| 160/160 [00:03<00:00, 42.86it/s]

✓ Inference completed docs: 80
Total predicted entities: 2182
Saved predictions to: C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_two_pass_2025to2026.json


## Error inspection on predictions
 1) **Per-label** precision/recall/F1 using exact span match.
 2) Overlap-based **confusion matrix** (best IoU match) to see label confusions.
 3) **Boundary error report**: correct label but wrong offsets (plus "near misses" within ±k chars).
 4) Most frequent **false-positive** strings per label (helps refine filters).

These tools are meant for iterative improvement (thresholds, filters, span handling).

In [22]:
import copy
import re
import pandas as pd
from collections import defaultdict, Counter


# ----------------------------
# Helpers (spans, IoU, indexing)
# ----------------------------
def norm_span(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def ent_key(ent):
    # inclusive end_idx per your format
    return (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]), str(ent["label"]))


def as_span(ent):
    return (int(ent["start_idx"]), int(ent["end_idx"]))  # inclusive


def overlap_len(a_start, a_end, b_start, b_end):
    left = max(a_start, b_start)
    right = min(a_end, b_end)
    return max(0, right - left + 1)


def iou(a_start, a_end, b_start, b_end):
    inter = overlap_len(a_start, a_end, b_start, b_end)
    if inter == 0:
        return 0.0
    a_len = a_end - a_start + 1
    b_len = b_end - b_start + 1
    return inter / (a_len + b_len - inter)


def build_index(entities):
    """
    idx[loc] = list[(start,end,ent)] sorted by start
    """
    idx = defaultdict(list)
    for e in entities:
        s, eend = as_span(e)
        loc = str(e["location"])
        idx[loc].append((s, eend, e))
    for loc in idx:
        idx[loc].sort(key=lambda x: x[0])
    return idx


def best_overlap_match(gold_ent, pred_candidates, min_iou=0.1):
    gs, ge = as_span(gold_ent)
    best = None
    best_iou = 0.0
    best_ol = 0

    for ps, pe, pent in pred_candidates:
        ol = overlap_len(gs, ge, ps, pe)
        if ol == 0:
            continue
        score = iou(gs, ge, ps, pe)
        if score < min_iou:
            continue
        if (score > best_iou) or (score == best_iou and ol > best_ol):
            best = pent
            best_iou = score
            best_ol = ol

    return best, best_iou, best_ol


# ----------------------------
# Flatten gold + pred
# ----------------------------
def flatten_gold(dev_data):
    gold = defaultdict(list)
    for pmid, article in dev_data.items():
        for e in article["entities"]:
            gold[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return gold


def flatten_pred(predictions):
    pred = defaultdict(list)
    for pmid, obj in predictions.items():
        for e in obj.get("entities", []):
            pred[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return pred


# ----------------------------
# (1) Per-label PRF (exact match)
# ----------------------------
def per_label_prf(gold_by_pmid, pred_by_pmid, labels):
    gold_sets = {lab: set() for lab in labels}
    pred_sets = {lab: set() for lab in labels}

    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            gold_sets[e["label"]].add((pmid,) + ent_key(e))

    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            pred_sets[e["label"]].add((pmid,) + ent_key(e))

    rows = []
    for lab in labels:
        g = gold_sets[lab]
        p = pred_sets[lab]
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)

        prec = tp / (tp + fp + 1e-12)
        rec = tp / (tp + fn + 1e-12)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)

        rows.append({
            "label": lab,
            "gold": len(g),
            "pred": len(p),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": prec,
            "recall": rec,
            "f1": f1,
        })

    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)


# ----------------------------
# (2) Confusion matrix (overlap-based)
# ----------------------------
def confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1):
    conf = pd.DataFrame(0, index=labels + ["<NONE>"], columns=labels + ["<NONE>"], dtype=int)

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        used_pred = set()
        for g in gold_ents:
            loc = g["location"]
            best, _, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            g_lab = g["label"]

            if best is None:
                conf.loc[g_lab, "<NONE>"] += 1
            else:
                bkey = (best["start_idx"], best["end_idx"], best["location"], best["label"], best.get("text_span", ""))
                used_pred.add(bkey)
                conf.loc[g_lab, best["label"]] += 1

        gold_idx = build_index(gold_ents)
        for p in pred_ents:
            pkey = (p["start_idx"], p["end_idx"], p["location"], p["label"], p.get("text_span", ""))
            if pkey in used_pred:
                continue
            loc = p["location"]
            best_gold, _, _ = best_overlap_match(p, gold_idx.get(loc, []), min_iou=min_iou)
            if best_gold is None:
                conf.loc["<NONE>", p["label"]] += 1

    return conf


# ----------------------------
# (3) Boundary report (same label overlap but offsets differ)
# ----------------------------
def boundary_report(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1, near_k=3):
    exact_tp = Counter()
    boundary_mismatch = Counter()
    near_miss = Counter()

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        pred_exact = set((pmid,) + ent_key(e) for e in pred_ents)

        for g in gold_ents:
            lab = g["label"]
            gk = (pmid,) + ent_key(g)

            if gk in pred_exact:
                exact_tp[lab] += 1
                continue

            loc = g["location"]
            best, _, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            if best is None:
                continue

            if best["label"] == lab:
                boundary_mismatch[lab] += 1
                if (abs(best["start_idx"] - g["start_idx"]) <= near_k) and (
                        abs(best["end_idx"] - g["end_idx"]) <= near_k):
                    near_miss[lab] += 1

    rows = []
    for lab in labels:
        tp = exact_tp[lab]
        bm = boundary_mismatch[lab]
        nm = near_miss[lab]
        denom = tp + bm
        rate = bm / (denom + 1e-12)
        rows.append({
            "label": lab,
            "exact_TP": tp,
            "boundary_mismatch_same_label": bm,
            "boundary_error_rate": rate,
            f"near_miss_within_±{near_k}": nm,
        })

    return pd.DataFrame(rows).sort_values("boundary_error_rate", ascending=False).reset_index(drop=True)


# ----------------------------
# (4) FP patterns
# ----------------------------
def fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15):
    gold_exact = set()
    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            gold_exact.add((pmid,) + ent_key(e))

    fp_by_label = defaultdict(Counter)
    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            pk = (pmid,) + ent_key(e)
            if pk in gold_exact:
                continue
            fp_by_label[e["label"]][norm_span(e.get("text_span", ""))] += 1

    for lab, counter in sorted(fp_by_label.items(), key=lambda x: sum(x[1].values()), reverse=True):
        total = sum(counter.values())
        print(f"\n=== Top FP strings for label: {lab} (total FP={total}) ===")
        for span, c in counter.most_common(top_n):
            print(f"{c:>4}  {span if span else '<EMPTY>'}")

## EVALUATION
Official-style evaluation (macro + micro)

In [23]:
LEGAL_ENTITY_LABELS = [
    "anatomical location", "animal", "bacteria", "biomedical technique",
    "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
    "human", "microbiome", "statistical technique"
]

def remove_duplicated_entities_copy(predictions):
    preds = copy.deepcopy(predictions)
    removed_count = 0
    for pmid in list(preds.keys()):
        seen = set()
        deduped = []
        for ent in preds[pmid]["entities"]:
            key = (ent["start_idx"], ent["end_idx"], ent["location"], ent["label"])
            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        preds[pmid]["entities"] = deduped
    return preds, removed_count

def remove_overlapping_entities_copy(predictions):
    """
    Keep longest in each overlap cluster per location.
    IMPORTANT: inclusive spans => overlap if next.start <= current_end
    """
    preds = copy.deepcopy(predictions)
    removed_count = 0

    for pmid in list(preds.keys()):
        original_len = len(preds[pmid]["entities"])
        groups = {"title": [], "abstract": []}
        for ent in preds[pmid]["entities"]:
            groups[str(ent["location"])].append(ent)

        keep_keys = set()
        for loc, group in groups.items():
            group = sorted(group, key=lambda e: int(e["start_idx"]))

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                s = int(ent["start_idx"])
                e = int(ent["end_idx"])
                if not cluster:
                    cluster = [ent]
                    current_end = e
                else:
                    # inclusive overlap => <=
                    if s <= current_end:
                        cluster.append(ent)
                        current_end = max(current_end, e)
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = e
            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = max(clust, key=lambda x: int(x["end_idx"]) - int(x["start_idx"]))
                keep_keys.add((int(longest["start_idx"]), int(longest["end_idx"]), str(longest["location"]), str(longest["label"])))

        deduped = []
        for ent in preds[pmid]["entities"]:
            k = (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]), str(ent["label"]))
            if k in keep_keys:
                deduped.append(ent)
                keep_keys.remove(k)
        preds[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    return preds, removed_count

def evaluate_ner_official(predictions, ground_truth_dev_data):
    """
    Ground truth must be dev_data-like (pmid -> {entities:[...]})
    """
    preds, dup_removed = remove_duplicated_entities_copy(predictions)
    preds, ov_removed = remove_overlapping_entities_copy(preds)

    ground_truth_NER = {}
    count_gold = Counter()
    for pmid, article in ground_truth_dev_data.items():
        gt_list = []
        for e in article["entities"]:
            lab = str(e["label"])
            if lab not in LEGAL_ENTITY_LABELS:
                continue
            entry = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["text_span"]), lab)
            gt_list.append(entry)
            count_gold[lab] += 1
        ground_truth_NER[pmid] = gt_list

    count_pred = Counter()
    count_tp = Counter()

    for pmid, obj in preds.items():
        for e in obj.get("entities", []):
            lab = str(e["label"])
            if lab not in LEGAL_ENTITY_LABELS:
                continue
            count_pred[lab] += 1
            entry = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["text_span"]), lab)
            if entry in set(ground_truth_NER.get(pmid, [])):
                count_tp[lab] += 1

    # micro
    total_gold = sum(count_gold.values())
    total_pred = sum(count_pred.values())
    total_tp = sum(count_tp.values())

    micro_p = total_tp / (total_pred + 1e-10)
    micro_r = total_tp / (total_gold + 1e-10)
    micro_f1 = 2 * (micro_p * micro_r) / (micro_p + micro_r + 1e-10)

    # macro (over labels present in gold)
    labels = list(count_gold.keys())
    macro_p = 0.0
    macro_r = 0.0
    macro_f1 = 0.0
    for lab in labels:
        p = count_tp[lab] / (count_pred[lab] + 1e-10)
        r = count_tp[lab] / (count_gold[lab] + 1e-10)
        f1 = 2 * (p * r) / (p + r + 1e-10)
        macro_p += p
        macro_r += r
        macro_f1 += f1

    n = max(1, len(labels))
    macro_p /= n
    macro_r /= n
    macro_f1 /= n

    meta = {
        "dup_removed": dup_removed,
        "overlap_removed": ov_removed,
        "count_gold": count_gold,
        "count_pred": count_pred,
        "count_tp": count_tp,
    }
    return macro_p, macro_r, macro_f1, micro_p, micro_r, micro_f1, meta


# ----------------------------
# RUN REPORTS
# ----------------------------
gold_by_pmid = flatten_gold(dev_data)                  # <-- official gold source
pred_by_pmid = flatten_pred(predictions_two_pass)

ALL_LABELS = sorted(set(
    [e["label"] for pmid in gold_by_pmid for e in gold_by_pmid[pmid]] +
    [e["label"] for pmid in pred_by_pmid for e in pred_by_pmid[pmid]]
))

df_prf = per_label_prf(gold_by_pmid, pred_by_pmid, ALL_LABELS)
print("\n=== Per-label Precision / Recall / F1 (exact span) ===")
print(df_prf.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

conf = confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1)
print("\n=== Confusion Matrix (rows=gold, cols=pred, overlap-based) ===")
print(conf.to_string())

df_boundary = boundary_report(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1, near_k=3)
print("\n=== Boundary Errors (same label overlap but offsets differ) ===")
print(df_boundary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15)

macro_p, macro_r, macro_f1, micro_p, micro_r, micro_f1, meta = evaluate_ner_official(predictions_two_pass, dev_data)

print("\n" + "="*60)
print("BERT NER - OFFICIAL-STYLE RESULTS")
print("="*60)
print("Macro-averaged Metrics:")
print(f"  Macro-Precision: {macro_p:.4f}")
print(f"  Macro-Recall:    {macro_r:.4f}")
print(f"  Macro-F1 Score:  {macro_f1:.4f}")
print("\nMicro-averaged Metrics:")
print(f"  Micro-Precision: {micro_p:.4f}")
print(f"  Micro-Recall:    {micro_r:.4f}")
print(f"  Micro-F1 Score:  {micro_f1:.4f}")
print("-"*60)
print(f"Removed duplicates: {meta['dup_removed']}")
print(f"Removed overlaps:   {meta['overlap_removed']}")
print("="*60)


=== Per-label Precision / Recall / F1 (exact span) ===
                label  gold  pred  tp  fp  fn  precision  recall     f1
                human   192   187 174  13  18     0.9305  0.9062 0.9182
           microbiome   231   209 200   9  31     0.9569  0.8658 0.9091
                  DDF   793   687 637  50 156     0.9272  0.8033 0.8608
               animal   152   141 121  20  31     0.8582  0.7961 0.8259
                 drug    75    77  60  17  15     0.7792  0.8000 0.7895
  anatomical location   169   170 130  40  39     0.7647  0.7692 0.7670
             bacteria   183   157 120  37  63     0.7643  0.6557 0.7059
   dietary supplement    64    46  38   8  26     0.8261  0.5937 0.6909
 biomedical technique   139   115  84  31  55     0.7304  0.6043 0.6614
             chemical   366   254 199  55 167     0.7835  0.5437 0.6419
statistical technique    35    45  25  20  10     0.5556  0.7143 0.6250
                 gene    63    59  35  24  28     0.5932  0.5556 0.5738
        

## Analysis: Entity Distribution by Label
- how many gold mentions exist in dev
- how many mentions your system predicts

 This helps spot systematic under/over-prediction:
 - predicted << gold => recall bottleneck for that label
- predicted >> gold => precision bottleneck for that label

In [24]:
# Entity Distribution by Label (Gold vs Pred)
pred_label_counts = Counter()
for pmid, pred in predictions_two_pass.items():
    for entity in pred["entities"]:
        pred_label_counts[entity["label"]] += 1

gold_label_counts = Counter()
for pmid, article in dev_data.items():
    for entity in article["entities"]:
        gold_label_counts[entity["label"]] += 1

print("\nEntity Distribution by Label:")
print("=" * 60)
print(f"{'Label':<25} {'Gold':<10} {'Predicted':<10}")
print("-" * 60)

all_labels = set(gold_label_counts.keys()) | set(pred_label_counts.keys())
for label in sorted(all_labels):
    print(f"{label:<25} {gold_label_counts[label]:<10} {pred_label_counts[label]:<10}")

print("-" * 60)
print(f"{'TOTAL':<25} {sum(gold_label_counts.values()):<10} {sum(pred_label_counts.values()):<10}")
print("=" * 60)


Entity Distribution by Label:
Label                     Gold       Predicted 
------------------------------------------------------------
DDF                       793        687       
anatomical location       169        170       
animal                    152        141       
bacteria                  183        157       
biomedical technique      139        115       
chemical                  366        254       
dietary supplement        64         46        
drug                      75         77        
food                      59         35        
gene                      63         59        
human                     192        187       
microbiome                231        209       
statistical technique     35         45        
------------------------------------------------------------
TOTAL                     2521       2182      
